# TomatoGuard — Phase 1: Train, quantize & calibrate the tomato leaf model

**Before you run (Kaggle settings, right sidebar):**
1. **Accelerator:** GPU T4 x2 (or P100).
2. **Internet: ON** (needed once for `pip` and to download the ImageNet-pretrained MobileNetV3 weights).
3. **Add Data → search "PlantVillage Dataset"** (by *abdallahalidev*) and add it. The notebook finds the `color` folder automatically.
4. **Run All.** Expect roughly 20–30 minutes.

**Outputs** (in `/kaggle/working`, also zipped as `tomatoguard_ml_artifacts.zip`):
`tomato_mobilenetv3_fp32.pth`, `tomato_mobilenetv3_fp32.onnx`, `tomato_mobilenetv3_int8.onnx`,
`classes.json`, `calibration.json`, `metrics.json`, `confusion_matrix.png`, `coverage_curve.png`, `splits.csv`.

**Honesty note:** PlantVillage has many near-duplicate photos of the same leaf, so the clean test score will be very high.
That is *not* field accuracy. The stress tests and out-of-distribution check at the end are there to show the gap.

In [ ]:
# --- 1. Setup & config -------------------------------------------------------
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript"], check=False)

import os, re, json, time, random, shutil, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from PIL import Image, ImageFilter, ImageEnhance
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import onnxruntime as ort

warnings.filterwarnings("ignore")

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
OUT.mkdir(parents=True, exist_ok=True)

IMG = 224
BATCH = 64
EPOCHS_HEAD = 3        # phase A: classifier head only
EPOCHS_FT = 12         # phase B: full network fine-tuning
LR_HEAD = 1e-3
LR_FT = 3e-4
PRETRAINED = True
TARGET_SEL_ACC = 0.95  # accuracy we want when the app chooses to answer
NUM_WORKERS = min(4, os.cpu_count() or 2)
ROOT_OVERRIDE = os.environ.get("TG_ROOT")  # only for local testing

print("device:", DEVICE, "| torch", torch.__version__, "| ort", ort.__version__, "| workers", NUM_WORKERS)
if DEVICE == "cpu":
    print("WARNING: no GPU detected. Training will be very slow. Enable a GPU accelerator in Kaggle settings.")

In [ ]:
# --- 2. Find dataset, build stratified 70/15/15 split --------------------------
CLASSES = [
    dict(id="bacterial_spot",         name="Bacterial Spot",          key="bacterialspot"),
    dict(id="early_blight",           name="Early Blight",            key="earlyblight"),
    dict(id="late_blight",            name="Late Blight",             key="lateblight"),
    dict(id="leaf_mold",              name="Leaf Mold",               key="leafmold"),
    dict(id="septoria_leaf_spot",     name="Septoria Leaf Spot",      key="septoria"),
    dict(id="spider_mites",           name="Spider Mites",            key="spidermite"),
    dict(id="target_spot",            name="Target Spot",             key="targetspot"),
    dict(id="mosaic_virus",           name="Mosaic Virus",            key="mosaic"),
    dict(id="yellow_leaf_curl_virus", name="Yellow Leaf Curl Virus",  key="yellowleafcurl"),
    dict(id="healthy",                name="Healthy",                 key="healthy"),
]
K = len(CLASSES)
norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())

def find_root():
    if ROOT_OVERRIDE:
        return Path(ROOT_OVERRIDE)
    cands = []
    for dp, dn, _ in os.walk("/kaggle/input"):
        if any(norm(d).startswith("tomato") and "lateblight" in norm(d) for d in dn):
            cands.append(Path(dp)); dn[:] = []   # don't descend into class folders
    if not cands:
        raise FileNotFoundError("No Tomato___Late_blight folder under /kaggle/input. Did you add the PlantVillage dataset?")
    good = [c for c in cands if "color" in str(c).lower() and "segment" not in str(c).lower() and "gray" not in str(c).lower()]
    return (good or cands)[0]

root = find_root()
print("dataset root:", root)
all_dirs = sorted(d for d in root.iterdir() if d.is_dir())
tomato_dirs = [d for d in all_dirs if norm(d.name).startswith("tomato")]
other_dirs = [d for d in all_dirs if d not in tomato_dirs]
for c in CLASSES:
    m = [d for d in tomato_dirs if c["key"] in norm(d.name)]
    assert len(m) == 1, f"class {c['id']} matched {m}"
    c["folder"] = m[0].name
print(f"{len(tomato_dirs)} tomato folders, {len(other_dirs)} other-crop folders")

EXT = {".jpg", ".jpeg", ".png"}
rows = []
for i, c in enumerate(CLASSES):
    for p in sorted((root / c["folder"]).iterdir()):
        if p.suffix.lower() in EXT:
            rows.append((str(p), i))
df = pd.DataFrame(rows, columns=["path", "label"])

tr, tmp = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=tmp.label, random_state=SEED)
df["split"] = "train"; df.loc[va.index, "split"] = "val"; df.loc[te.index, "split"] = "test"
df.to_csv(OUT / "splits.csv", index=False)

summary = df.assign(cls=df.label.map(lambda i: CLASSES[i]["id"])).groupby(["cls", "split"]).size().unstack()
print(summary[["train", "val", "test"]]); print("\nTotals:", df.split.value_counts().to_dict())

In [ ]:
# --- 3. Datasets, augmentation, degradations ----------------------------------
# NOTE: images are fed to the model as RGB in [0,1]. ImageNet normalisation is INSIDE the model,
# so the web app only has to resize to 224x224 and divide by 255.
train_tf = T.Compose([
    T.RandomResizedCrop(IMG, scale=(0.5, 1.0), ratio=(0.8, 1.25)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(25),
    T.ColorJitter(0.4, 0.4, 0.3, 0.03),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.5))], p=0.25),
    T.ToTensor(),
])
eval_tf = T.Compose([T.Resize((IMG, IMG)), T.ToTensor()])

KINDS = ["blur", "dark", "bright", "lowres", "zoomout", "noise"]

def degrade(img, kind, rng):
    w, h = img.size
    if kind == "blur":
        return img.filter(ImageFilter.GaussianBlur(rng.uniform(2, 6)))
    if kind == "dark":
        return ImageEnhance.Brightness(img).enhance(rng.uniform(0.15, 0.45))
    if kind == "bright":
        return ImageEnhance.Brightness(img).enhance(rng.uniform(1.8, 3.0))
    if kind == "lowres":
        s = rng.randint(32, 64)
        return img.resize((s, s), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    if kind == "zoomout":   # leaf small in frame, random background
        f = rng.uniform(0.35, 0.6)
        small = img.resize((int(w * f), int(h * f)))
        bg = Image.new("RGB", (w, h), tuple(rng.randint(40, 200) for _ in range(3)))
        bg.paste(small, (rng.randint(0, w - small.width), rng.randint(0, h - small.height)))
        return bg
    if kind == "noise":
        a = np.asarray(img).astype(np.float32)
        a += np.random.default_rng(rng.randint(0, 10**9)).normal(0, rng.uniform(15, 40), a.shape)
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    raise ValueError(kind)

class LeafDS(Dataset):
    def __init__(self, paths, labels, tf, kind=None):
        self.paths, self.labels, self.tf, self.kind = list(paths), list(labels), tf, kind
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.kind:
            rng = random.Random(SEED * 100003 + i)       # deterministic per image
            k = rng.choice(KINDS) if self.kind == "random" else self.kind
            img = degrade(img, k, rng)
        return self.tf(img), int(self.labels[i])

def make_loader(paths, labels, tf, shuffle=False, kind=None, bs=BATCH):
    return DataLoader(LeafDS(paths, labels, tf, kind), batch_size=bs, shuffle=shuffle, drop_last=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

train_loader = make_loader(tr.path, tr.label, train_tf, shuffle=True)
val_loader   = make_loader(va.path, va.label, eval_tf)
test_loader  = make_loader(te.path, te.label, eval_tf)

# quick visual sanity check of augmentations / degradations
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
sample = Image.open(tr.path.iloc[0]).convert("RGB")
ax[0, 0].imshow(sample); ax[0, 0].set_title("original")
for a_, k in zip(ax.flat[1:7], KINDS):
    a_.imshow(degrade(sample, k, random.Random(1))); a_.set_title(k)
ax[1, 3].imshow(train_tf(sample).permute(1, 2, 0)); ax[1, 3].set_title("train aug")
for a_ in ax.flat: a_.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# --- 4. Model + two-phase training -------------------------------------------
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class TomatoNet(nn.Module):
    """MobileNetV3-Small with ImageNet normalisation baked in (input: RGB float in [0,1])."""
    def __init__(self, n_classes, pretrained=True):
        super().__init__()
        w = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.mobilenet_v3_small(weights=w)
        self.backbone.classifier[3] = nn.Linear(self.backbone.classifier[3].in_features, n_classes)
        self.register_buffer("mean", torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(STD).view(1, 3, 1, 1))
    def forward(self, x):
        return self.backbone((x - self.mean) / self.std)

model = TomatoNet(K, PRETRAINED).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")

# class weights (sqrt inverse frequency, mean-normalised) — mosaic virus has far fewer images than YLCV
counts = np.bincount(tr.label, minlength=K)
cw = (counts.sum() / (K * counts)) ** 0.5
cw = cw / cw.mean()
print("class weights:", dict(zip([c["id"] for c in CLASSES], cw.round(2))))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEVICE), label_smoothing=0.05)

use_amp = DEVICE == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

@torch.no_grad()
def torch_logits(m, loader):
    m.eval(); L, Y = [], []
    for x, y in loader:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            out = m(x.to(DEVICE, non_blocking=True))
        L.append(out.float().cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(L), np.concatenate(Y)

best = {"f1": -1.0, "epoch": None, "phase": None}
history = []

def fit(phase, epochs, lr, train_features):
    for p in model.backbone.features.parameters():
        p.requires_grad = train_features
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if train_features else None
    for ep in range(1, epochs + 1):
        model.train()
        if not train_features:
            model.backbone.features.eval()   # keep BatchNorm statistics fixed while only the head trains
        tot, n, t0 = 0.0, 0, time.time()
        for x, y in tqdm(train_loader, leave=False, desc=f"{phase} ep{ep}"):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y); n += len(y)
        if sched: sched.step()
        lg, yy = torch_logits(model, val_loader)
        acc = accuracy_score(yy, lg.argmax(1)); f1 = f1_score(yy, lg.argmax(1), average="macro")
        history.append(dict(phase=phase, epoch=ep, train_loss=tot / n, val_acc=acc, val_macro_f1=f1))
        star = ""
        if f1 > best["f1"]:
            best.update(f1=f1, epoch=ep, phase=phase)
            torch.save(model.state_dict(), OUT / "tomato_mobilenetv3_fp32.pth"); star = "  <- best"
        print(f"[{phase} {ep:02d}/{epochs}] loss {tot/n:.4f} | val acc {acc:.4f} | val macroF1 {f1:.4f} | {time.time()-t0:.0f}s{star}")

fit("head", EPOCHS_HEAD, LR_HEAD, train_features=False)
fit("finetune", EPOCHS_FT, LR_FT, train_features=True)
print("\nbest:", best)

model.load_state_dict(torch.load(OUT / "tomato_mobilenetv3_fp32.pth", map_location=DEVICE))
model.eval()

In [ ]:
# --- 5. Evaluate FP32 PyTorch model on the held-out TEST set ------------------
lg_test, y_test = torch_logits(model, test_loader)
pred = lg_test.argmax(1)
torch_test_acc = float(accuracy_score(y_test, pred))
torch_test_f1 = float(f1_score(y_test, pred, average="macro"))
names = [c["name"] for c in CLASSES]
report = classification_report(y_test, pred, target_names=names, output_dict=True, digits=4)
print(classification_report(y_test, pred, target_names=names, digits=4))
print(f"TEST accuracy {torch_test_acc:.4f} | macro-F1 {torch_test_f1:.4f}")

cm = confusion_matrix(y_test, pred)
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(K)); ax.set_yticks(range(K))
ax.set_xticklabels(names, rotation=45, ha="right"); ax.set_yticklabels(names)
for i in range(K):
    for j in range(K):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cmn[i, j] > 0.5 else "black", fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix — FP32 model, test set (counts)")
plt.colorbar(im, fraction=0.046); plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=150); plt.show()

In [ ]:
# --- 6. Export FP32 ONNX ------------------------------------------------------
model.cpu().eval()
fp32_onnx = OUT / "tomato_mobilenetv3_fp32.onnx"
dummy = torch.rand(1, 3, IMG, IMG)
kw = dict(input_names=["input"], output_names=["logits"],
          dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, do_constant_folding=True)
try:
    torch.onnx.export(model, dummy, str(fp32_onnx), dynamo=False, **kw)   # classic exporter (most compatible)
except TypeError:
    torch.onnx.export(model, dummy, str(fp32_onnx), **kw)

import onnx
onnx.checker.check_model(onnx.load(str(fp32_onnx)))
xb = torch.stack([eval_tf(Image.open(p).convert("RGB")) for p in te.path.iloc[:8]])
with torch.no_grad(): ref = model(xb).numpy()
got = ort.InferenceSession(str(fp32_onnx), providers=["CPUExecutionProvider"]).run(None, {"input": xb.numpy()})[0]
print(f"ONNX vs PyTorch max |diff| = {np.abs(ref - got).max():.2e}  (should be ~1e-5 or smaller)")
print("saved", fp32_onnx.name, f"{fp32_onnx.stat().st_size/1e6:.2f} MB")
model.to(DEVICE)

In [ ]:
# --- 7. INT8 static quantisation (two calibration methods; pick on VALIDATION) --
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType, CalibrationMethod
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
    pre = OUT / "_pre.onnx"; quant_pre_process(str(fp32_onnx), str(pre)); q_src = pre
except Exception as e:
    print("quant_pre_process skipped:", repr(e)[:120]); q_src = fp32_onnx

rng_np = np.random.default_rng(SEED)
calib = []
for k in range(K):                                   # 20 clean TRAIN images per class
    ps = tr[tr.label == k].path.values
    for p in rng_np.choice(ps, size=min(20, len(ps)), replace=False):
        calib.append(eval_tf(Image.open(p).convert("RGB")).unsqueeze(0).numpy())
print("calibration images:", len(calib))

class Reader(CalibrationDataReader):
    def __init__(self, arrs): self.it = iter([{"input": a} for a in arrs])
    def get_next(self): return next(self.it, None)

def ort_logits(path, loader):
    s = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    L, Y = [], []
    for x, y in loader:
        L.append(s.run(None, {"input": x.numpy()})[0]); Y.append(y.numpy())
    return np.concatenate(L), np.concatenate(Y)

q_candidates = {}
for name, cmeth in {"minmax": CalibrationMethod.MinMax, "percentile": CalibrationMethod.Percentile}.items():
    outp = OUT / f"_int8_{name}.onnx"
    try:
        quantize_static(str(q_src), str(outp), Reader(calib), quant_format=QuantFormat.QDQ,
                        activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8, per_channel=True,
                        calibrate_method=cmeth, extra_options={"ActivationSymmetric": False, "WeightSymmetric": True})
        q_candidates[name] = outp
    except Exception as e:
        print(f"quantisation '{name}' failed:", repr(e)[:200])
assert q_candidates, "All quantisation attempts failed — see messages above."

lg_val_fp32, y_val = ort_logits(fp32_onnx, val_loader)
val_f1_fp32 = float(f1_score(y_val, lg_val_fp32.argmax(1), average="macro"))
val_f1_q = {}
for name, pth in q_candidates.items():
    lg, _ = ort_logits(pth, val_loader)
    val_f1_q[name] = float(f1_score(y_val, lg.argmax(1), average="macro"))
    print(f"INT8 [{name:10s}] val macro-F1 {val_f1_q[name]:.4f}   (FP32 ONNX: {val_f1_fp32:.4f})")
best_q = max(val_f1_q, key=val_f1_q.get)
int8_onnx = OUT / "tomato_mobilenetv3_int8.onnx"
shutil.copy(q_candidates[best_q], int8_onnx)
for p in list(q_candidates.values()) + [OUT / "_pre.onnx"]:
    if Path(p).exists(): Path(p).unlink()
print("chosen INT8 calibration:", best_q)

In [ ]:
# --- 8. FP32 vs INT8: accuracy, size, latency (TEST set) ----------------------
def latency_ms(path, n=100):
    so = ort.SessionOptions(); so.intra_op_num_threads = 1
    s = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    x = np.random.rand(1, 3, IMG, IMG).astype(np.float32)
    for _ in range(10): s.run(None, {"input": x})
    ts = []
    for _ in range(n):
        t = time.perf_counter(); s.run(None, {"input": x}); ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts)), float(np.percentile(ts, 95))

cmp = {}
test_logits = {}
for tag, pth in [("fp32_onnx", fp32_onnx), ("int8_onnx", int8_onnx)]:
    lg, yy = ort_logits(pth, test_loader); test_logits[tag] = lg
    med, p95 = latency_ms(pth)
    cmp[tag] = dict(test_accuracy=float(accuracy_score(yy, lg.argmax(1))),
                    test_macro_f1=float(f1_score(yy, lg.argmax(1), average="macro")),
                    size_mb=round(pth.stat().st_size / 1e6, 3),
                    latency_ms_median_1thread=round(med, 1), latency_ms_p95_1thread=round(p95, 1))
cmp["prediction_agreement_fp32_vs_int8"] = float((test_logits["fp32_onnx"].argmax(1) == test_logits["int8_onnx"].argmax(1)).mean())
print(pd.DataFrame({k: v for k, v in cmp.items() if isinstance(v, dict)}).T.to_string())
print("agreement FP32 vs INT8 predictions:", round(cmp["prediction_agreement_fp32_vs_int8"], 4))
print("(Latency is on the Kaggle CPU with 1 thread — NOT a phone. Measure on a real phone in the PWA.)")

# Ship INT8 only if it did not lose more than 1 point of validation macro-F1
DEPLOY = "int8" if (val_f1_fp32 - val_f1_q[best_q]) <= 0.01 else "fp32"
deploy_path = int8_onnx if DEPLOY == "int8" else fp32_onnx
print(f"\nDEPLOY MODEL -> {deploy_path.name}")
if DEPLOY == "fp32":
    print("INT8 lost more than 1 F1 point on validation; the app should use the FP32 ONNX until quantisation is improved.")

In [ ]:
# --- 9. "I'm not sure": temperature scaling + confidence threshold (VALIDATION only) --
# The threshold is chosen on clean validation images PLUS degraded copies of them (blur, dark, bright,
# low-res, zoomed-out, noise) so that it reflects behaviour on imperfect phone photos, not just lab images.
val_deg_loader = make_loader(va.path, va.label, eval_tf, kind="random")
lg_v_clean, y_v = ort_logits(deploy_path, val_loader)
lg_v_deg, _ = ort_logits(deploy_path, val_deg_loader)
LG = np.concatenate([lg_v_clean, lg_v_deg]); YY = np.concatenate([y_v, y_v])
print(f"val clean acc {accuracy_score(y_v, lg_v_clean.argmax(1)):.4f} | val degraded acc {accuracy_score(y_v, lg_v_deg.argmax(1)):.4f}")

def softmax_np(z, Tmp=1.0):
    z = z / Tmp; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def fit_temperature(logits, labels):
    lg_t, y_t = torch.tensor(logits), torch.tensor(labels)
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(lg_t / logT.exp(), y_t); loss.backward(); return loss
    opt.step(closure)
    return float(logT.exp())

TEMP = fit_temperature(LG, YY)
probs = softmax_np(LG, TEMP); conf = probs.max(1); correct = (probs.argmax(1) == YY)
print(f"temperature T = {TEMP:.3f}")

def sel_stats(conf, correct, thr):
    ans = conf >= thr
    return float(ans.mean()), (float(correct[ans].mean()) if ans.any() else float("nan"))

grid = np.round(np.arange(0.30, 0.995, 0.01), 2)
curve = [(t, *sel_stats(conf, correct, t)) for t in grid]
ok = [t for t, cov, acc in curve if acc >= TARGET_SEL_ACC]
THRESH = float(min(ok)) if ok else 0.90
THRESH = max(THRESH, 0.50)           # never answer with less than a 50% calibrated probability
HIGH_THRESH = float(min(0.99, max(0.90, THRESH + 0.10)))
print(f"\nCHOSEN THRESHOLD = {THRESH:.2f}  (lowest value giving >= {TARGET_SEL_ACC:.0%} accuracy-when-answering on clean+degraded validation)")
print("\n thr   coverage  acc_when_answering")
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, THRESH]:
    cov, acc = sel_stats(conf, correct, t); print(f" {t:.2f}   {cov:7.3f}   {acc:7.3f}")

ts_, cv_, ac_ = zip(*curve)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ts_, cv_, label="coverage (share of photos answered)"); ax.plot(ts_, ac_, label="accuracy when answering")
ax.axvline(THRESH, color="red", ls="--", label=f"chosen threshold {THRESH:.2f}"); ax.axhline(TARGET_SEL_ACC, color="gray", ls=":")
ax.set_xlabel("confidence threshold"); ax.set_ylim(0.5, 1.01); ax.legend(loc="lower left"); ax.grid(alpha=.3)
ax.set_title("Validation (clean + degraded)"); plt.tight_layout(); plt.savefig(OUT / "coverage_curve.png", dpi=150); plt.show()

In [ ]:
# --- 10. Final honest evaluation: TEST clean, TEST stress, and OTHER-CROP images --
def evaluate_loader(loader, label="", ood=False):
    lg, yy = ort_logits(deploy_path, loader)
    p = softmax_np(lg, TEMP); c = p.max(1); pr = p.argmax(1)
    if ood:
        return dict(n=int(len(c)), accepted_as_tomato=float((c >= THRESH).mean()), rejected=float((c < THRESH).mean())), c
    corr = pr == yy
    cov, acc = sel_stats(c, corr, THRESH)
    return dict(n=int(len(yy)), raw_accuracy=float(corr.mean()), coverage=cov, accuracy_when_answering=acc), c

final = {}
final["test_clean"], _ = evaluate_loader(test_loader)

sub = te.sample(n=min(600, len(te)), random_state=SEED)
final["test_stress"] = {}
for k in KINDS:
    final["test_stress"][k], _ = evaluate_loader(make_loader(sub.path, sub.label, eval_tf, kind=k))
final["test_stress"]["random_mix"], _ = evaluate_loader(make_loader(sub.path, sub.label, eval_tf, kind="random"))

print(f"Threshold used: {THRESH:.2f}, T={TEMP:.3f}\n")
print(pd.DataFrame({"test_clean": final["test_clean"], **{f"stress_{k}": v for k, v in final["test_stress"].items()}}).T.round(4).to_string())

# Other crops (potato, pepper, corn, grape, ...) pushed through the tomato classifier
rng_py = random.Random(SEED)
ood_paths, ood_folder = [], []
for d in other_dirs:
    fs = sorted(p for p in d.iterdir() if p.suffix.lower() in EXT)
    for p in rng_py.sample(fs, min(30, len(fs))):
        ood_paths.append(str(p)); ood_folder.append(d.name)
res, ood_conf = evaluate_loader(make_loader(ood_paths, [-1] * len(ood_paths), eval_tf), ood=True)
final["other_crops"] = res
ood_df = pd.DataFrame({"folder": ood_folder, "accepted": ood_conf >= THRESH})
per_folder = ood_df.groupby("folder").accepted.mean().sort_values(ascending=False)
final["other_crops"]["worst_folders_accepted_as_tomato"] = {k: round(float(v), 3) for k, v in per_folder.head(8).items()}
print(f"\nOTHER CROPS ({res['n']} images): {res['rejected']:.1%} rejected by the confidence threshold, "
      f"{res['accepted_as_tomato']:.1%} would still get a tomato diagnosis.")
print(per_folder.head(8).round(3).to_string())
print("\n=> If many other-crop photos are accepted, the confidence threshold alone is NOT a tomato/not-tomato detector."
      "\n   That is exactly why Phase 2 (a separate Tomato / Not-Tomato detector) is needed.")

In [ ]:
# --- 11. Save metadata + zip --------------------------------------------------
classes_out = {
    "classes": [dict(index=i, id=c["id"], name=c["name"], folder=c["folder"]) for i, c in enumerate(CLASSES)],
    "preprocessing": dict(input_name="input", output_name="logits", layout="NCHW", size=[IMG, IMG],
                          resize="direct resize of the whole photo to 224x224 (no crop)",
                          pixel_range="RGB float32 in [0,1] (value/255); ImageNet normalisation is inside the model"),
}
calibration = dict(model_file=deploy_path.name, temperature=TEMP, threshold=THRESH, high_confidence_threshold=HIGH_THRESH,
                   rule="probs = softmax(logits / temperature); if max(probs) < threshold -> say 'I'm not sure'",
                   target_accuracy_when_answering=TARGET_SEL_ACC)
metrics = dict(
    dataset=dict(name="PlantVillage (tomato subset)", n_images=int(len(df)),
                 split_sizes=df.split.value_counts().to_dict(), seed=SEED, root=str(root)),
    training=dict(arch="MobileNetV3-Small (ImageNet init)", epochs_head=EPOCHS_HEAD, epochs_finetune=EPOCHS_FT,
                  lr_head=LR_HEAD, lr_finetune=LR_FT, batch=BATCH, label_smoothing=0.05, best=best, history=history),
    fp32_pytorch_test=dict(accuracy=torch_test_acc, macro_f1=torch_test_f1, per_class=report),
    fp32_vs_int8=cmp, int8_calibration_method=best_q, deployed=DEPLOY,
    uncertainty=dict(temperature=TEMP, threshold=THRESH, high_confidence_threshold=HIGH_THRESH),
    final_evaluation=final,
    caveats=["PlantVillage images are lab-style photos of single leaves on plain backgrounds; PlantVillage contains many "
             "near-duplicates, so the random split inflates test scores. These numbers are NOT field accuracy.",
             "Latency was measured on a Kaggle CPU (1 thread), not on a phone.",
             "Threshold was chosen on validation data (clean + synthetically degraded), not on real farmer photos.",
             "Non-tomato rejection by threshold alone is unreliable; use the separate tomato/not-tomato detector."])
(OUT / "classes.json").write_text(json.dumps(classes_out, indent=2))
(OUT / "calibration.json").write_text(json.dumps(calibration, indent=2))
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))

files = ["tomato_mobilenetv3_fp32.pth", "tomato_mobilenetv3_fp32.onnx", "tomato_mobilenetv3_int8.onnx",
         "classes.json", "calibration.json", "metrics.json", "confusion_matrix.png", "coverage_curve.png", "splits.csv"]
with zipfile.ZipFile(OUT / "tomatoguard_ml_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(OUT / f, f)
for f in files + ["tomatoguard_ml_artifacts.zip"]:
    print(f"{f:36s} {(OUT / f).stat().st_size/1e6:8.2f} MB")

print("\n===== HEADLINE NUMBERS (paste these back to me) =====")
print(f"FP32 test acc / macro-F1 : {cmp['fp32_onnx']['test_accuracy']:.4f} / {cmp['fp32_onnx']['test_macro_f1']:.4f}")
print(f"INT8 test acc / macro-F1 : {cmp['int8_onnx']['test_accuracy']:.4f} / {cmp['int8_onnx']['test_macro_f1']:.4f}   deployed={DEPLOY}")
print(f"sizes MB (fp32 / int8)   : {cmp['fp32_onnx']['size_mb']} / {cmp['int8_onnx']['size_mb']}")
print(f"threshold / temperature  : {THRESH:.2f} / {TEMP:.3f}")
print(f"test coverage / sel.acc  : {final['test_clean']['coverage']:.3f} / {final['test_clean']['accuracy_when_answering']:.3f}")
print(f"other crops rejected     : {final['other_crops']['rejected']:.1%}")

In [ ]:
# --- 11. Save metadata (Arabic) + zip -----------------------------------------
ARABIC_RESULTS = {
    "bacterial_spot": {
        "name": "التبقع البكتيري",
        "speech": "ممكن تكون الطماطم مصابة بالتبقع البكتيري. بص على باقي الزرع وشوف لو فيه نفس الأعراض."
    },
    "early_blight": {
        "name": "اللفحة المبكرة",
        "speech": "ممكن تكون الطماطم مصابة باللفحة المبكرة. بص على باقي الزرع وشوف لو فيه نفس الأعراض."
    },
    "healthy": {
        "name": "الورقة شكلها سليم",
        "speech": "الورقة شكلها سليم، ومفيش علامات واضحة على وجود مرض."
    },
    "late_blight": {
        "name": "اللفحة المتأخرة",
        "speech": "ممكن تكون الطماطم مصابة باللفحة المتأخرة. بص على باقي الزرع وشوف لو فيه نفس الأعراض."
    },
    "leaf_mold": {
        "name": "عفن الأوراق",
        "speech": "ممكن تكون الطماطم مصابة بعفن الأوراق. بص على باقي الزرع وشوف لو فيه نفس الأعراض."
    },
    "mosaic_virus": {
        "name": "فيروس الموزاييك",
        "speech": "ممكن تكون الطماطم مصابة بفيروس الموزاييك. بص على باقي الزرع وشوف لو فيه نفس الأعراض."
    },
    "septoria_leaf_spot": {
        "name": "تبقع الأوراق",
        "speech": "ممكن تكون الطماطم مصابة بتبقع الأوراق. بص على باقي الزرع وشوف لو فيه نفس الأعراض."
    },
    "spider_mites": {
        "name": "العنكبوت الأحمر",
        "speech": "ممكن يكون فيه عنكبوت أحمر على الطماطم. بص على باقي الزرع وشوف لو فيه نفس الأعراض."
    },
    "target_spot": {
        "name": "تبقع الهدف",
        "speech": "ممكن تكون الطماطم مصابة بتبقع الهدف. بص على باقي الزرع وشوف لو فيه نفس الأعراض."
    },
    "yellow_leaf_curl_virus": {
        "name": "فيروس تجعد الأوراق الأصفر",
        "speech": "ممكن تكون الطماطم مصابة بفيروس تجعد الأوراق الأصفر. بص على باقي الزرع وشوف لو فيه نفس الأعراض."
    },
}
assert set(ARABIC_RESULTS) == {c["id"] for c in CLASSES}, "ARABIC_RESULTS keys must match the class ids"

# Screens that are not a disease result (same dialect). Review the wording before the demo.
ARABIC_MESSAGES = {
    "uncertain": {
        "title": "مش متأكد",
        "speech": "مش قادر أحدد المشكلة بشكل مؤكد. صوّر ورقة طماطم واحدة واضحة في ضوء كويس وجرّب تاني."
    },
    "not_tomato": {
        "title": "دي مش شكل ورقة طماطم",
        "speech": "الصورة دي مش شكل ورقة طماطم. صوّر ورقة طماطم وجرّب تاني."
    },
    "bad_image": {
        "title": "مش شايف الورقة كويس",
        "speech": "مش شايف الورقة بوضوح. صوّر تاني في ضوء كويس."
    },
    "confidence_high": "ثقة عالية",
    "confidence_medium": "ثقة متوسطة",
    "extension_hint": "لو المشكلة بتزيد أو لسه مش متأكد، كلّم المرشد الزراعي.",
}

classes_out = {
    "classes": [dict(index=i, id=c["id"], name=c["name"], folder=c["folder"],
                     name_ar=ARABIC_RESULTS[c["id"]]["name"],
                     speech_ar=ARABIC_RESULTS[c["id"]]["speech"]) for i, c in enumerate(CLASSES)],
    "messages_ar": ARABIC_MESSAGES,
    "preprocessing": dict(input_name="input", output_name="logits", layout="NCHW", size=[IMG, IMG],
                          resize="direct resize of the whole photo to 224x224 (no crop)",
                          pixel_range="RGB float32 in [0,1] (value/255); ImageNet normalisation is inside the model"),
}
calibration = dict(model_file=deploy_path.name, temperature=TEMP, threshold=THRESH, high_confidence_threshold=HIGH_THRESH,
                   rule="probs = softmax(logits / temperature); if max(probs) < threshold -> say 'I'm not sure'",
                   target_accuracy_when_answering=TARGET_SEL_ACC)
metrics = dict(
    dataset=dict(name="PlantVillage (tomato subset)", n_images=int(len(df)),
                 split_sizes=df.split.value_counts().to_dict(), seed=SEED, root=str(root)),
    training=dict(arch="MobileNetV3-Small (ImageNet init)", epochs_head=EPOCHS_HEAD, epochs_finetune=EPOCHS_FT,
                  lr_head=LR_HEAD, lr_finetune=LR_FT, batch=BATCH, label_smoothing=0.05, best=best, history=history),
    fp32_pytorch_test=dict(accuracy=torch_test_acc, macro_f1=torch_test_f1, per_class=report),
    fp32_vs_int8=cmp, int8_calibration_method=best_q, deployed=DEPLOY,
    uncertainty=dict(temperature=TEMP, threshold=THRESH, high_confidence_threshold=HIGH_THRESH),
    final_evaluation=final,
    caveats=["PlantVillage images are lab-style photos of single leaves on plain backgrounds; PlantVillage contains many "
             "near-duplicates, so the random split inflates test scores. These numbers are NOT field accuracy.",
             "Latency was measured on a Kaggle CPU (1 thread), not on a phone.",
             "Threshold was chosen on validation data (clean + synthetically degraded), not on real farmer photos.",
             "Non-tomato rejection by threshold alone is unreliable; use the separate tomato/not-tomato detector."])
# ensure_ascii=False + utf-8 keeps the Arabic readable in the JSON files
(OUT / "classes.json").write_text(json.dumps(classes_out, indent=2, ensure_ascii=False), encoding="utf-8")
(OUT / "calibration.json").write_text(json.dumps(calibration, indent=2), encoding="utf-8")
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float), encoding="utf-8")

files = ["tomato_mobilenetv3_fp32.pth", "tomato_mobilenetv3_fp32.onnx", "tomato_mobilenetv3_int8.onnx",
         "classes.json", "calibration.json", "metrics.json", "confusion_matrix.png", "coverage_curve.png", "splits.csv"]
with zipfile.ZipFile(OUT / "tomatoguard_ml_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(OUT / f, f)
for f in files + ["tomatoguard_ml_artifacts.zip"]:
    print(f"{f:36s} {(OUT / f).stat().st_size/1e6:8.2f} MB")

# ---- What the farmer will see / hear, per class (Arabic) ----
print("\n===== النتائج بالعربي (ما يظهر ويُقرأ بصوت عالٍ لكل حالة) =====")
for c in CLASSES:
    r = ARABIC_RESULTS[c["id"]]
    print(f"- {r['name']}\n    🔊 {r['speech']}")
for k in ("uncertain", "not_tomato", "bad_image"):
    print(f"- {ARABIC_MESSAGES[k]['title']}\n    🔊 {ARABIC_MESSAGES[k]['speech']}")

# ---- Test examples: model output -> Arabic answer, using the real threshold ----
def arabic_answer(logits_row):
    p = softmax_np(logits_row[None, :], TEMP)[0]; i = int(p.argmax()); cf = float(p[i])
    if cf < THRESH:
        m = ARABIC_MESSAGES["uncertain"]; return m["title"], m["speech"], cf
    r = ARABIC_RESULTS[CLASSES[i]["id"]]
    return r["name"], r["speech"], cf

print("\n===== أمثلة حقيقية من مجموعة الاختبار =====")
lg_ex, y_ex = ort_logits(deploy_path, make_loader(te.path.iloc[:200], te.label.iloc[:200], eval_tf))
shown = set()
for lg_row, y_true in zip(lg_ex, y_ex):
    if int(y_true) in shown: continue
    shown.add(int(y_true))
    title, speech, cf = arabic_answer(lg_row)
    print(f"[الحقيقة: {CLASSES[int(y_true)]['name']}]  ->  {title}  (ثقة {cf:.0%})\n    🔊 {speech}")
    if len(shown) == 4: break

print("\n===== HEADLINE NUMBERS (paste these back to me) =====")
print(f"FP32 test acc / macro-F1 : {cmp['fp32_onnx']['test_accuracy']:.4f} / {cmp['fp32_onnx']['test_macro_f1']:.4f}")
print(f"INT8 test acc / macro-F1 : {cmp['int8_onnx']['test_accuracy']:.4f} / {cmp['int8_onnx']['test_macro_f1']:.4f}   deployed={DEPLOY}")
print(f"sizes MB (fp32 / int8)   : {cmp['fp32_onnx']['size_mb']} / {cmp['int8_onnx']['size_mb']}")
print(f"threshold / temperature  : {THRESH:.2f} / {TEMP:.3f}")
print(f"test coverage / sel.acc  : {final['test_clean']['coverage']:.3f} / {final['test_clean']['accuracy_when_answering']:.3f}")
print(f"other crops rejected     : {final['other_crops']['rejected']:.1%}")

In [ ]:
# --- 12. Package the model for the phone (offline bundle) + self-test ----------
import hashlib, datetime

BUNDLE = OUT / "tomatoguard_mobile_bundle"
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
(BUNDLE / "models").mkdir(parents=True); (BUNDLE / "demo_images").mkdir()

# 1) model + labels/Arabic text + threshold, all under stable file names
model_dst = BUNDLE / "models" / "tomato_classifier.onnx"
shutil.copy(deploy_path, model_dst)
shutil.copy(OUT / "classes.json", BUNDLE / "models" / "classes.json")
cal = json.loads((OUT / "calibration.json").read_text()); cal["model_file"] = model_dst.name
(BUNDLE / "models" / "calibration.json").write_text(json.dumps(cal, indent=2))

sha = hashlib.sha256(model_dst.read_bytes()).hexdigest()
info = dict(version=datetime.date.today().isoformat(), precision=("INT8" if DEPLOY == "int8" else "FP32"),
            size_mb=round(model_dst.stat().st_size / 1e6, 3), sha256=sha, input_shape=[1, 3, IMG, IMG],
            test_accuracy=cmp[f"{DEPLOY}_onnx"]["test_accuracy"], test_macro_f1=cmp[f"{DEPLOY}_onnx"]["test_macro_f1"],
            note="Accuracy is on PlantVillage lab-style images, not field photos.")
(BUNDLE / "models" / "model_info.json").write_text(json.dumps(info, indent=2))

# 2) demo images: best correct test photo per class + blurry / dark / other-crop examples
lg_dep, y_dep = ort_logits(deploy_path, test_loader)
p_dep = softmax_np(lg_dep, TEMP); conf_dep = p_dep.max(1); ok_dep = p_dep.argmax(1) == y_dep
te_paths = te.path.tolist()
for k, c in enumerate(CLASSES):
    idx = [i for i in range(len(y_dep)) if y_dep[i] == k and ok_dep[i]]
    if idx:
        j = max(idx, key=lambda i: conf_dep[i])
        Image.open(te_paths[j]).convert("RGB").save(BUNDLE / "demo_images" / f"leaf_{c['id']}.jpg", quality=92)
healthy_k = [c["id"] for c in CLASSES].index("healthy")
base = Image.open(BUNDLE / "demo_images" / "leaf_healthy.jpg")
degrade(base, "blur", random.Random(3)).save(BUNDLE / "demo_images" / "demo_blurry.jpg", quality=92)
degrade(base, "dark", random.Random(3)).save(BUNDLE / "demo_images" / "demo_dark.jpg", quality=92)
degrade(base, "zoomout", random.Random(3)).save(BUNDLE / "demo_images" / "demo_leaf_far_away.jpg", quality=92)
Image.open(ood_paths[0]).convert("RGB").save(BUNDLE / "demo_images" / f"demo_other_crop.jpg", quality=92)

# 3) SELF-TEST: load ONLY the bundle files, exactly as the phone app will, and run every demo image
sess = ort.InferenceSession(str(model_dst), providers=["CPUExecutionProvider"])
cls_json = json.loads((BUNDLE / "models" / "classes.json").read_text(encoding="utf-8"))
cal_b = json.loads((BUNDLE / "models" / "calibration.json").read_text())
def bundle_predict(path):
    x = eval_tf(Image.open(path).convert("RGB")).unsqueeze(0).numpy()
    z = sess.run(None, {"input": x})[0]
    p = softmax_np(z, cal_b["temperature"])[0]; i = int(p.argmax()); cf = float(p[i])
    if cf < cal_b["threshold"]:
        return "UNCERTAIN", cf, cls_json["messages_ar"]["uncertain"]["title"]
    return cls_json["classes"][i]["id"], cf, cls_json["classes"][i]["name_ar"]

print("===== SELF-TEST of the bundle (what the phone will do) =====")
for f in sorted((BUNDLE / "demo_images").glob("*.jpg")):
    cid, cf, ar = bundle_predict(f)
    print(f"{f.name:34s} -> {cid:24s} conf {cf:5.2f} | {ar}")
print("\nNote: 'demo_other_crop' is NOT a tomato. If it got a tomato label above, the threshold alone failed -> we need the tomato/not-tomato detector.")

# 4) README + zip
(BUNDLE / "README.txt").write_text(
"""TomatoGuard model bundle
models/tomato_classifier.onnx  - the model that runs ON THE PHONE (no server)
models/classes.json            - class order, Arabic names/speech, preprocessing spec
models/calibration.json        - temperature + 'I'm not sure' threshold
models/model_info.json         - version, size, sha256, metrics
demo_images/                   - photos for the demo (healthy, diseases, blurry, dark, far away, other crop)

Preprocessing: RGB -> resize whole photo to 224x224 -> divide by 255 -> NCHW float32. (Normalisation is inside the model.)
Post-processing: probs = softmax(logits / temperature); if max(probs) < threshold -> 'not sure'.
Copy the 'models' folder into the web app at public/models/ .
""", encoding="utf-8")
zp = shutil.make_archive(str(OUT / "tomatoguard_mobile_bundle"), "zip", BUNDLE)
print(f"\nBUNDLE READY: {Path(zp).name}  ({Path(zp).stat().st_size/1e6:.2f} MB)")
print("Model on phone:", model_dst.name, f"{model_dst.stat().st_size/1e6:.2f} MB, precision {info['precision']}")
print("Download: Kaggle right sidebar -> Output -> tomatoguard_mobile_bundle.zip  (use 'Save Version' -> 'Save & Run All' so it is kept).")